# CS 3110/5110: Data Privacy
## Homework 8

In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
import random
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def laplace_mech_vec(vec, sensitivity, epsilon):
    return [v + np.random.laplace(loc=0, scale=sensitivity / epsilon) for v in vec]

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)
            for v in vec]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

In [2]:
# preserves epsilon-differential privacy
def above_threshold(queries, df, T, epsilon):
    T_hat = T + np.random.laplace(loc=0, scale = 2/epsilon)
    
    for idx, q in enumerate(queries):
        nu_i = np.random.laplace(loc=0, scale = 4/epsilon)
        if q(df) + nu_i >= T_hat:
            return idx
    return -1 # the index of the last element

## Question 1 (20 points)

Implement a function `above_10000` that releases the **value** of the first query in a sequence of queries whose value is above 10000. Your function should have a **total** privacy cost equal to the privacy parameter $\epsilon$ passed in when it is called.

In [38]:
def make_query(status):
    def q(df):
        return len(df[df['Marital Status'] == status])
    return q

def above_10000(queries, epsilon):
    T = 1000
    idx = above_threshold(queries, adult, T=T, epsilon=epsilon/2)
    qeuery_value = queries[idx](adult)
    noisy_value = laplace_mech(qeuery_value, sensitivity=1, epsilon=1/2)
    return noisy_value
    

queries = [make_query(status) for status in adult['Marital Status'].unique()]
print(f"above_10000 #1: {above_10000(queries, 100)}")
print(f"above_10000 #2: {above_10000(queries, 1)}")
print(f"above_10000 #3: {above_10000(queries, .01)}")

above_10000 #1: 10682.589011232752
above_10000 #2: 10682.210167254532
above_10000 #3: 10680.845065938433


In [21]:
# TEST CASE

results = [above_10000(queries, 1.0) for _ in range(100)]
print(np.mean(results))
assert np.mean(results) > 9900
assert np.mean(results) < 11000

10682.69031841585


## Question 2 (10 points)
In 2-3 sentences, argue informally (via the definition of the sparse vector technique, post-processing, and sequential composition), that your implementation of `above_10000` has a total privacy cost of $\epsilon$.

YOUR ANSWER HERE

## Question 3 (20 points)

Implement a function `bounded_all_above_10000` that releases the **value** of **$c$ queries** in a sequence of queries whose value is above 10000 (where $c$ is an analyst-provided parameter limiting the number of returned results). Your function should have a **total privacy cost** bounded by its parameter $\epsilon$.

In [ ]:
def bounded_all_above_10000(queries, c, epsilon):
    T = 1000
    queries_temp_in = queries
    noisy_vals = []
    epsilon_c = epsilon/c
    for _ in range(c):
        if not queries_temp_in:
            break
        
        idx = above_threshold(queries, adult, T=T, epsilon=epsilon_c/2)

        if idx == -1    :
            break

        true_val = queries_temp_in[idx](adult)
        noisy_val = laplace_mech(true_val, sensitivity=1, epsilon=epsilon_c/2)
        noisy_vals.append(noisy_val)

        queries_temp_in = queries_temp_in[idx + 1:]

    
    return noisy_vals

queries = [make_query(status) for status in adult['Marital Status'].unique()]
print(f"bounded_all_above_10000 #1: {bounded_all_above_10000(queries, 3, 100)}")
print(f"bounded_all_above_10000 #2: {bounded_all_above_10000(queries, 3, 1)}")
print(f"bounded_all_above_10000 #3: {bounded_all_above_10000(queries, 3, .01)}")

bounded_all_above_10000 #1: [10683.035223088838, 14976.023052449702, 4443.00281103898]
bounded_all_above_10000 #2: [10668.49747489838, 14975.143036083013, 4446.817137069637]
bounded_all_above_10000 #3: [10523.905379444093, 15069.959879347944, 4671.303073701987]


In [49]:
# TEST CASE

results = [bounded_all_above_10000(queries, 2, 1.0) for _ in range(100)]
results_1 = [r[0] for r in results]
results_2 = [r[1] for r in results]

assert np.mean(results_1) > 9900
assert np.mean(results_1) < 11000
assert np.mean(results_2) > 14000
assert np.mean(results_2) < 15500

## Question 4 (10 points)

In 2-3 sentences, argue informally that your implementation of `bounded_all_above_10000` has privacy cost bounded by $\epsilon$.

YOUR ANSWER HERE

## Question 5 (30 points)

Implement a function `mean_age` that computes the mean age of participants in the `adult_data` dataset. Your function should have a **total** privacy cost of $\epsilon$. It should work as follows:

1. Compute an upper clipping parameter based on the data using a differentially private algorithm
2. Clip the data using the clipping parameter
3. Use `laplace_mech` to release a differentially private mean of the clipped data

*Hint*: Use the sparse vector technique (`above_threshold`) to compute the clipping parameter. Consider using a sequence of queries that looks like `df.clip(lower=0, upper=b).sum() - df.clip(lower=0, upper=b+1).sum()`.

*Hint*: Be careful of sensitivities and set the scale of the noise accordingly!

In [53]:
bs = list(range(0, 200, 10))
df = adult['Age']
b_lower = 0

def make_query(b):
    def q(df):
        # Return positive val if plateau is reached
        sum1 = df['Age'].clip(lower=0, upper=b).sum() # sens of b, not 1
        sum2 = df['Age'].clip(lower=0, upper=b-1).sum() # sens of b, not 1
        return sum2 - sum1 # sens of 1, taking diff between 2 summations. 
    return q

def mean_age(epsilon):
    queries = [make_query(b) for b in range(1, 1000)]
    b = above_threshold(queries, adult,T=0, epsilon=epsilon/2)
    adults_clip = adult["Age"].clip(lower=0, upper=b)
    true_mean = adults_clip["Age"].mean()
    noisy_mean = laplace_mech(true_mean, sensitivity=b, epsilon=epsilon/2)
    return noisy_mean
    
for epsilon in [0.001, 0.01, 0.1, 0.5, 1, 10]:
    print(f"epsilon: {epsilon}, mean age: {mean_age(epsilon)}")

KeyError: 'Age'

In [50]:
# TEST CASE
results = [mean_age(1.0) for _ in range(100)]
assert np.mean(results) > 38
assert np.mean(results) < 39

0
0
3
0
0
1
0
1
1
0
1
0
1
2
1
1
8
3
0
2
1
2
0
1
0
3
3
0
2
9
2
1
1
17
3
11
0
8
0
2
2
1
0
0
0
0
0
2
2
1
2
1
6
1
1
1
0
3
0
0
0
6
3
0
0
4
1
2
1
0
6
3
1
3
0
3
4
0
0
0
1
1
1
2
1
0
1
0
1
0
0
0
0
0
1
3
0
1
0
2


TypeError: unsupported operand type(s) for +: 'NoneType' and 'NoneType'

## Question 6 (10 points)

In 3-5 sentences, describe your approach to implementing `mean_age` and argue informally that your implementation has privacy cost bounded by $\epsilon$.

YOUR ANSWER HERE

## Question 7 (20 points)

Write an algorithm to compute the (approximate) maximum of a given column of the adult dataset. Your solution should satisfy `epsilon` differential privacy.

*Hint*: you might want to use AboveThreshold; you might need to test several options for building your queries and for the threshold

In [ ]:
def max_of_col(column, epsilon):
    # YOUR CODE HERE
    raise NotImplementedError()

max_of_col('Age', 1.0)

In [ ]:
# TEST CASE
cols = ['Age', 'Capital Gain', 'fnlwgt']

for c in cols:
    true_val = adult[c].max()
    trials = [max_of_col(c, 10.0) for _ in range(20)]
    errors = [pct_error(true_val, t) for t in trials]
    print('Median error for column ' + c + ':', np.median(errors))
    assert np.median(errors) > 0
    assert np.median(errors) < 100

## Question 8 (10 points)

In 3-5 sentences, describe your approach to implementing `max_of_col` and argue that your approach has total privacy cost bounded by $\epsilon$.

YOUR ANSWER HERE